# Preprocessing Data: retail_store_sales.csv
## Untuk Analisis Apriori dan FP-Growth

Notebook ini melakukan preprocessing dataset `retail_store_sales.csv` agar sesuai dengan format dataset contoh `UTS_Apriori dan FP-Growth.xlsx`.

**Output Excel memiliki 3 Sheet (sama seperti contoh UTS):**
1. `Datasets` — Format market basket (Item(s), Item 1, Item 2, ...)
2. `Tranform to one-hot` — Transformasi one-hot encoding
3. `Association Rules to ExampleSet` — Hasil analisis Apriori

**Langkah-langkah:**
1. Import Library
2. Load Dataset
3. Eksplorasi Data Awal (EDA)
4. Data Cleaning
5. Transformasi ke Format Market Basket
6. Transformasi One-Hot Encoding
7. Analisis Association Rules (Apriori)
8. Export ke Excel (3 Sheet)

## 1. Import Library

In [ ]:
%pip install pandas openpyxl mlxtend

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

## 2. Load Dataset

In [ ]:
# Load dataset retail_store_sales
df = pd.read_csv('retail_store_sales.csv')

print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print(f'\nKolom dataset:')
print(df.columns.tolist())
df.head(10)

## 3. Eksplorasi Data Awal (EDA)

In [ ]:
# Informasi dataset
print('=== Info Dataset ===')
df.info()

In [ ]:
# Statistik deskriptif
print('=== Statistik Deskriptif ===')
df.describe()

In [ ]:
# Cek missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct.round(2)})
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Cek duplikat
print('=== Duplikat ===')
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')

In [ ]:
# Cek nilai unik untuk kolom kategorikal
print('=== Nilai Unik ===')
print(f"Jumlah Transaction ID unik: {df['Transaction ID'].nunique()}")
print(f"Jumlah Customer ID unik: {df['Customer ID'].nunique()}")
print(f"\nKategori produk ({df['Category'].nunique()} kategori):")
print(df['Category'].value_counts())
print(f"\nJumlah Item unik: {df['Item'].nunique()}")
print(f"\nPayment Method:")
print(df['Payment Method'].value_counts())
print(f"\nLocation:")
print(df['Location'].value_counts())

## 4. Data Cleaning

In [ ]:
# Simpan ukuran awal
print(f'Ukuran dataset awal: {df.shape}')

# 4.1 Hapus baris duplikat
df_clean = df.drop_duplicates()
print(f'Setelah hapus duplikat: {df_clean.shape}')

# 4.2 Hapus baris yang memiliki missing value pada kolom Item
print(f"\nMissing values pada kolom 'Item' sebelum cleaning: {df_clean['Item'].isnull().sum()}")
df_clean = df_clean.dropna(subset=['Item'])
print(f"Setelah hapus missing pada 'Item': {df_clean.shape}")

# 4.3 Hapus baris yang memiliki missing value pada kolom Quantity dan Total Spent
df_clean = df_clean.dropna(subset=['Quantity', 'Total Spent'])
print(f"Setelah hapus missing pada 'Quantity' & 'Total Spent': {df_clean.shape}")

print(f'\n=== Hasil Cleaning ===')
print(f'Ukuran dataset bersih: {df_clean.shape}')
print(f'Data yang dihapus: {df.shape[0] - df_clean.shape[0]} baris')
print(f'Total missing values tersisa pada kolom Item: {df_clean["Item"].isnull().sum()}')

In [ ]:
# Verifikasi data bersih
print('=== Verifikasi Data Bersih ===')
print(df_clean.isnull().sum())
print(f'\nJumlah duplikat: {df_clean.duplicated().sum()}')
df_clean.head(10)

## 5. Transformasi ke Format Market Basket (Sheet 1: Datasets)

Pada dataset retail ini, setiap baris = 1 transaksi dengan 1 item.
Untuk analisis Apriori/FP-Growth, item dikelompokkan berdasarkan **Customer ID**
sehingga setiap baris = daftar semua item unik yang dibeli oleh 1 customer.

Format mengikuti contoh UTS:
- `Item(s)` = jumlah item unik
- `Item 1`, `Item 2`, ... = nama item

In [ ]:
# Grouping item UNIK berdasarkan Customer ID
transactions = df_clean.groupby('Customer ID')['Item'].apply(lambda x: sorted(set(x))).reset_index()
transactions.columns = ['Customer ID', 'Items']
transactions['Item(s)'] = transactions['Items'].apply(len)

print(f'Jumlah keranjang: {len(transactions)}')
print(f'Rata-rata item/keranjang: {transactions["Item(s)"].mean():.2f}')
print(f'Max item/keranjang: {transactions["Item(s)"].max()}')
print(f'Min item/keranjang: {transactions["Item(s)"].min()}')

transactions.head()

In [ ]:
# Pecah list items menjadi kolom terpisah: Item(s), Item 1, Item 2, ...
max_items = transactions['Item(s)'].max()
item_columns = [f'Item {i+1}' for i in range(max_items)]

items_expanded = transactions['Items'].apply(pd.Series)
items_expanded.columns = item_columns[:items_expanded.shape[1]]

# Sheet 1: Datasets (format sama seperti UTS)
df_datasets = pd.concat([transactions[['Item(s)']], items_expanded], axis=1)

print(f'Shape: {df_datasets.shape}')
print(f'Kolom: Item(s), Item 1, Item 2, ..., Item {max_items}')
df_datasets.head(10)

In [ ]:
# Perbandingan format dengan contoh UTS
print('=== Perbandingan dengan Format UTS ===')
df_uts = pd.read_excel('UTS_Apriori dan FP-Growth.xlsx', sheet_name='Datasets')
print(f'\nContoh UTS (Datasets):')
print(f'  Shape  : {df_uts.shape}')
print(f'  Kolom  : {df_uts.columns.tolist()[:5]} ...')
print(df_uts.head(3).to_string())

print(f'\nHasil Preprocessing (Datasets):')
print(f'  Shape  : {df_datasets.shape}')
print(f'  Kolom  : {df_datasets.columns.tolist()[:5]} ...')
print(df_datasets.head(3).to_string())
print(f'\n✅ Format Sheet Datasets sudah sesuai!')

## 6. Transformasi One-Hot Encoding (Sheet 2: Tranform to one-hot)

Mengubah data transaksi menjadi format one-hot encoding (0/1) dimana:
- Setiap kolom = 1 item unik
- Setiap baris = 1 transaksi (customer)
- Nilai 1 = customer membeli item tersebut, 0 = tidak

In [ ]:
# Buat one-hot encoding dari transaksi
all_items = sorted(df_clean['Item'].unique())
print(f'Jumlah item unik: {len(all_items)}')

# Buat dictionary per customer
customer_items = df_clean.groupby('Customer ID')['Item'].apply(set).to_dict()

# Buat DataFrame one-hot
onehot_data = []
for cust_id in sorted(customer_items.keys()):
    row = {item: (1 if item in customer_items[cust_id] else 0) for item in all_items}
    onehot_data.append(row)

df_onehot = pd.DataFrame(onehot_data)

print(f'Shape one-hot: {df_onehot.shape}')
print(f'\nContoh:')
df_onehot.head()

In [ ]:
# Perbandingan dengan Sheet one-hot UTS
print('=== Perbandingan One-Hot ===')
df_uts_oh = pd.read_excel('UTS_Apriori dan FP-Growth.xlsx', sheet_name='Tranform to one-hot')
print(f'\nContoh UTS (one-hot):')
print(f'  Shape  : {df_uts_oh.shape}')
print(f'  Kolom  : {df_uts_oh.columns.tolist()[:5]} ...')

print(f'\nHasil Preprocessing (one-hot):')
print(f'  Shape  : {df_onehot.shape}')
print(f'  Kolom  : {df_onehot.columns.tolist()[:5]} ...')
print(f'\n✅ Format Sheet One-Hot sudah sesuai!')

## 7. Analisis Association Rules - Apriori (Sheet 3: Association Rules to ExampleSet)

Menjalankan algoritma Apriori untuk menemukan frequent itemsets dan association rules.

In [ ]:
# Jalankan Apriori dengan minimum support
df_onehot_bool = df_onehot.astype(bool)
frequent_itemsets = apriori(df_onehot_bool, min_support=0.3, use_colnames=True)

print(f'Jumlah frequent itemsets: {len(frequent_itemsets)}')
frequent_itemsets.head(10)

In [ ]:
# Generate Association Rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5, num_itemsets=len(frequent_itemsets))

# Format kolom agar mirip dengan contoh UTS
df_rules = pd.DataFrame({
    'Premises': rules['antecedents'].apply(lambda x: ', '.join(list(x))),
    'Premises Frequency': rules['antecedent support'].apply(lambda x: round(x * len(df_onehot))),
    'Conclusion': rules['consequents'].apply(lambda x: ', '.join(list(x))),
    'Conclusion Frequency': rules['consequent support'].apply(lambda x: round(x * len(df_onehot))),
    'Premise and Conclusion Frequency': rules['support'].apply(lambda x: round(x * len(df_onehot))),
    'Support': rules['support'].round(6),
    'Confidence': rules['confidence'].round(6),
    'Lift': rules['lift'].round(6),
    'Conviction': rules['conviction'].round(6),
    'Leverage': rules['leverage'].round(6)
})

# Sort by Confidence descending
df_rules = df_rules.sort_values('Confidence', ascending=False).reset_index(drop=True)

print(f'Jumlah Association Rules: {len(df_rules)}')
df_rules.head(15)

In [ ]:
# Perbandingan dengan Sheet Association Rules UTS
print('=== Perbandingan Association Rules ===')
df_uts_rules = pd.read_excel('UTS_Apriori dan FP-Growth.xlsx', sheet_name='Association Rules to ExampleSet')
print(f'\nContoh UTS (Association Rules):')
print(f'  Shape  : {df_uts_rules.shape}')
print(f'  Kolom  : {df_uts_rules.columns.tolist()}')

print(f'\nHasil Preprocessing (Association Rules):')
print(f'  Shape  : {df_rules.shape}')
print(f'  Kolom  : {df_rules.columns.tolist()}')
print(f'\n✅ Format Sheet Association Rules sudah sesuai!')

## 8. Export ke Excel (3 Sheet - Sama dengan Format UTS)

In [ ]:
# Export ke Excel dengan 3 sheet (format sama seperti UTS)
output_file = 'retail_store_sales_preprocessed.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: Datasets (market basket format)
    df_datasets.to_excel(writer, sheet_name='Datasets', index=False)
    
    # Sheet 2: Tranform to one-hot
    df_onehot.to_excel(writer, sheet_name='Tranform to one-hot', index=False)
    
    # Sheet 3: Association Rules to ExampleSet
    df_rules.to_excel(writer, sheet_name='Association Rules to ExampleSet', index=False)

print(f'✅ File berhasil disimpan: {output_file}')
print(f'\n   Sheet 1: Datasets                        ({df_datasets.shape[0]} baris x {df_datasets.shape[1]} kolom)')
print(f'   Sheet 2: Tranform to one-hot              ({df_onehot.shape[0]} baris x {df_onehot.shape[1]} kolom)')
print(f'   Sheet 3: Association Rules to ExampleSet   ({df_rules.shape[0]} baris x {df_rules.shape[1]} kolom)')

In [ ]:
# Ringkasan akhir
print('=' * 60)
print('RINGKASAN PREPROCESSING')
print('=' * 60)
print(f'Dataset awal            : {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Setelah cleaning        : {df_clean.shape[0]} baris x {df_clean.shape[1]} kolom')
print(f'Data dihapus            : {df.shape[0] - df_clean.shape[0]} baris')
print(f'Jumlah keranjang        : {len(df_datasets)} (per Customer ID)')
print(f'Jumlah item unik        : {len(all_items)}')
print(f'Max item/keranjang      : {df_datasets["Item(s)"].max()}')
print(f'Jumlah Association Rules : {len(df_rules)}')
print(f'\nOutput: {output_file}')
print(f'  Sheet 1: Datasets')
print(f'  Sheet 2: Tranform to one-hot')
print(f'  Sheet 3: Association Rules to ExampleSet')
print('=' * 60)